# 4. Query and Visualize Data

This notebook demonstrates DataJoint query operations and data visualization for the LC-MS pipeline.

In [ ]:
from lcms_demo.config import use_local_database

use_local_database()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from lcms_demo import subject, session, scan

## Basic Queries

### Fetch All Data

In [ ]:
# View subjects table
subject.Subject()

In [ ]:
# Fetch all subjects as a pandas DataFrame
subjects_df = subject.Subject.fetch(format="frame")
subjects_df

### Restriction (Filtering)

In [ ]:
# Filter samples by type
subject.Sample & "sample_type = 'plasma'"

In [ ]:
# Filter scans by retention time window
scan.Scan & "retention_time BETWEEN 5 AND 10"

### Projection (Selecting Columns)

In [ ]:
# Select and rename attributes
scan.Scan.proj(rt="retention_time", tic="total_ion_current")

### Join Operations

In [ ]:
# Join tables to get combined information
subject.Subject * subject.Sample

### Aggregation

In [ ]:
# Count scans per session
scan.Scan.aggr(session.Session, n_scans="count(*)")

---

## Data Visualization

### Total Ion Chromatogram (TIC)

The TIC shows the total ion current as a function of retention time.

In [ ]:
# Get the first session
first_session = session.Session.fetch(limit=1, as_dict=True)[0]
session_key = {k: first_session[k] for k in ['subject_id', 'sample_id', 'session_datetime']}

# Fetch scan data for this session
scans_df = (scan.Scan & session_key).fetch(
    'retention_time', 'total_ion_current',
    order_by='retention_time',
    format='frame'
).reset_index()

scans_df.head()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

ax.plot(scans_df['retention_time'], scans_df['total_ion_current'], 'b-', linewidth=0.8)
ax.fill_between(scans_df['retention_time'], scans_df['total_ion_current'], alpha=0.3)

ax.set_xlabel('Retention Time (min)')
ax.set_ylabel('Total Ion Current')
ax.set_title(f"Total Ion Chromatogram - {first_session['subject_id']} / {first_session['sample_id']}")
ax.ticklabel_format(style='scientific', axis='y', scilimits=(0,0))

plt.tight_layout()
plt.show()

### Base Peak Chromatogram (BPC)

The BPC shows the intensity of the most intense peak at each scan.

In [ ]:
# Fetch base peak data
bp_df = (scan.Scan & session_key).fetch(
    'retention_time', 'base_peak_mz', 'base_peak_intensity',
    order_by='retention_time',
    format='frame'
).reset_index()

fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

# Base peak intensity
axes[0].plot(bp_df['retention_time'], bp_df['base_peak_intensity'], 'g-', linewidth=0.8)
axes[0].fill_between(bp_df['retention_time'], bp_df['base_peak_intensity'], alpha=0.3, color='green')
axes[0].set_ylabel('Base Peak Intensity')
axes[0].set_title('Base Peak Chromatogram')
axes[0].ticklabel_format(style='scientific', axis='y', scilimits=(0,0))

# Base peak m/z
axes[1].scatter(bp_df['retention_time'], bp_df['base_peak_mz'], s=2, alpha=0.5, c='purple')
axes[1].set_xlabel('Retention Time (min)')
axes[1].set_ylabel('Base Peak m/z')
axes[1].set_title('Base Peak m/z vs Retention Time')

plt.tight_layout()
plt.show()

### Mass Spectrum

Display a single mass spectrum from the dataset.

In [ ]:
# Get spectrum for scan at peak TIC
peak_idx = scans_df['total_ion_current'].idxmax()
peak_scan = scans_df.iloc[peak_idx]

# Fetch the spectrum data
spectrum_key = {**session_key, 'scan_number': int(peak_scan.name) if hasattr(peak_scan.name, '__int__') else peak_idx + 1}

# Get scan number from the actual data
scan_numbers = (scan.Scan & session_key).fetch('scan_number', order_by='retention_time')
spectrum_key = {**session_key, 'scan_number': scan_numbers[peak_idx]}

spectrum = (scan.ScanSpectrum & spectrum_key).fetch1()
print(f"Spectrum at RT = {peak_scan['retention_time']:.2f} min (scan #{spectrum_key['scan_number']})")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

mz = spectrum['mz_array']
intensity = spectrum['intensity_array']

# Plot as stem plot for mass spectrum
ax.vlines(mz, 0, intensity, linewidth=0.5, color='blue')

ax.set_xlabel('m/z')
ax.set_ylabel('Intensity')
ax.set_title(f"Mass Spectrum at RT = {peak_scan['retention_time']:.2f} min")
ax.ticklabel_format(style='scientific', axis='y', scilimits=(0,0))

plt.tight_layout()
plt.show()

### Zoomed Spectrum Region

In [ ]:
# Zoom into a region around the base peak
base_peak_mz = mz[np.argmax(intensity)]
window = 50  # +/- 50 Da

mask = (mz >= base_peak_mz - window) & (mz <= base_peak_mz + window)

fig, ax = plt.subplots(figsize=(10, 4))

ax.vlines(mz[mask], 0, intensity[mask], linewidth=1, color='blue')

ax.set_xlabel('m/z')
ax.set_ylabel('Intensity')
ax.set_title(f"Mass Spectrum (m/z {base_peak_mz-window:.0f} - {base_peak_mz+window:.0f})")
ax.ticklabel_format(style='scientific', axis='y', scilimits=(0,0))

plt.tight_layout()
plt.show()

### Detected Peaks

Visualize the peaks detected by the PeakList computed table.

In [ ]:
# Check if peaks have been computed
peak_list = scan.PeakList & spectrum_key

if peak_list:
    peak_count = peak_list.fetch1('peak_count')
    peaks = (scan.PeakList.Peak & spectrum_key).fetch(format='frame').reset_index()
    print(f"Detected {peak_count} peaks")
    peaks.head(10)
else:
    print("No peaks detected yet. Run PeakList.populate() to detect peaks.")

In [ ]:
if peak_list:
    fig, ax = plt.subplots(figsize=(10, 4))
    
    # Plot full spectrum
    ax.vlines(mz, 0, intensity, linewidth=0.3, color='gray', alpha=0.5)
    
    # Overlay detected peaks
    ax.vlines(peaks['mz'], 0, peaks['intensity'], linewidth=1, color='red', label='Detected peaks')
    ax.scatter(peaks['mz'], peaks['intensity'], s=10, color='red', zorder=5)
    
    ax.set_xlabel('m/z')
    ax.set_ylabel('Intensity')
    ax.set_title(f"Detected Peaks ({peak_count} peaks)")
    ax.ticklabel_format(style='scientific', axis='y', scilimits=(0,0))
    ax.legend()
    
    plt.tight_layout()
    plt.show()

### Peak Statistics

In [ ]:
if peak_list:
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    
    # SNR distribution
    axes[0].hist(peaks['snr'], bins=30, edgecolor='black', alpha=0.7)
    axes[0].set_xlabel('Signal-to-Noise Ratio')
    axes[0].set_ylabel('Count')
    axes[0].set_title('Peak SNR Distribution')
    axes[0].axvline(peaks['snr'].median(), color='red', linestyle='--', label=f"Median: {peaks['snr'].median():.1f}")
    axes[0].legend()
    
    # m/z vs intensity
    scatter = axes[1].scatter(peaks['mz'], peaks['intensity'], c=peaks['snr'], s=10, cmap='viridis', alpha=0.7)
    axes[1].set_xlabel('m/z')
    axes[1].set_ylabel('Intensity')
    axes[1].set_title('Peak Intensity vs m/z')
    axes[1].ticklabel_format(style='scientific', axis='y', scilimits=(0,0))
    plt.colorbar(scatter, ax=axes[1], label='SNR')
    
    plt.tight_layout()
    plt.show()

## Summary Statistics

In [ ]:
print("Pipeline Summary")
print("=" * 40)
print(f"Subjects:      {len(subject.Subject()):>6}")
print(f"Samples:       {len(subject.Sample()):>6}")
print(f"Sessions:      {len(session.Session()):>6}")
print(f"Scans:         {len(scan.Scan()):>6}")
print(f"Spectra:       {len(scan.ScanSpectrum()):>6}")
print(f"Peak Lists:    {len(scan.PeakList()):>6}")